# Gait dataset speed summary
This notebook loads all `.mat` files under `/Users/markusgambietz/Downloads/Gait Dataset/` with `scipy.io.loadmat`, extracts right-side hip/knee/ankle angle means and force `Fx`/`Fy` means, saves each participant-speed trial individually to `reference_data/walking_processed/` (mirroring `reference_data/fukuchi_processed/`), and computes mean/variance across participants per speed into `reference_data/gait_speed_mean_var.npz` + per-metric CSVs.

In [ ]:
import glob
import os
import re
import numpy as np
import pandas as pd
import scipy.io

root_dir = '/Users/markusgambietz/Downloads/Gait Dataset'
mat_files = sorted(glob.glob(os.path.join(root_dir, '**', '*.mat'), recursive=True))
print(f'Found {len(mat_files)} .mat files')

def parse_speed_from_path(path):
    basename = os.path.basename(path)
    # Example: 1_60norm_data.mat -> 1.60
    m = re.search(r'(\d+)_(\d+)', basename)
    if m:
        return float(f'{m.group(1)}.{m.group(2)}')
    m = re.search(r'speed[_-]?(\d+\.?\d*)', path, re.IGNORECASE)
    if m:
        return float(m.group(1))
    parts = path.split(os.sep)
    for part in reversed(parts):
        if re.fullmatch(r'\d+\.?\d*', part):
            return float(part)
    return None

def get_nested(data, keys):
    cur = data
    for key in keys:
        if cur is None:
            return None
        if isinstance(cur, dict):
            if key in cur:
                cur = cur[key]
            elif key.lower() in cur:
                cur = cur[key.lower()]
            elif key.upper() in cur:
                cur = cur[key.upper()]
            else:
                return None
        else:
            return None
    return cur

def ensure_array(value):
    if value is None:
        return None
    return np.atleast_1d(np.asarray(value, dtype=float))

def resample_series(series, target_len):
    series = np.asarray(series, dtype=float)
    if len(series) == target_len:
        return series
    if len(series) < 2:
        return np.full(target_len, float(series[0]) if len(series) == 1 else np.nan)
    x_src = np.linspace(0.0, 1.0, len(series))
    x_dst = np.linspace(0.0, 1.0, target_len)
    return np.interp(x_dst, x_src, series)

# NOTE: 'ankle' has no 'FLEXION' key in this dataset -- only 'PLANTARFLEXION'/'INVERSION'.
# And the lab's raw 'Fx'/'Fy'/'Fz' axes don't line up with our Fx/Fy: verified by regressing
# the raw per-participant means against the already-published reference_data/gait_speed_mean_var.npz
# (slope=1.0, intercept=0.0 to float precision) that our Fx is the lab's raw Fy (anteroposterior)
# and our Fy is the lab's raw Fz (vertical) -- consistent with the OpenSim Y-up convention used
# for the simulated grf_x/grf_y elsewhere in this project.
joint_map = {
    'hip_flexion': ['HIP', 'FLEXION'],
    'knee_angle': ['KNEE', 'FLEXION'],
    'ankle_angle': ['ANKLE', 'PLANTARFLEXION'],
}
force_map = {
    'Fx': ['FORCE', 'Fy'],
    'Fy': ['FORCE', 'Fz'],
}
side_key = 'RIGHT'

# Individual-trial reference files, one per participant per speed, saved in the same
# format as reference_data/fukuchi_processed/*.mat (hip/knee/ankle in radians, knee
# flipped to flexion-negative, Fx/Fy in body weight) so downstream code (see
# data_exploration_main.ipynb, load_reference_subject_curves) can treat the reference
# dataset as "just another experiment" the same way it already does for Fukuchi running data.
walking_processed_dir = os.path.join('reference_data', 'walking_processed')
os.makedirs(walking_processed_dir, exist_ok=True)
n_saved = 0

speed_groups = {}
load_errors = []

for path in mat_files:
    speed = parse_speed_from_path(path)
    if speed is None:
        load_errors.append((path, 'speed not parsed'))
        continue

    try:
        data = scipy.io.loadmat(path, squeeze_me=True, simplify_cells=True)
    except Exception as exc:
        load_errors.append((path, f'load error: {exc}'))
        continue

    metrics = get_nested(data, ['metrics_normative'])
    if metrics is None:
        load_errors.append((path, 'missing metrics_normative'))
        continue

    side_data = get_nested(metrics, [side_key])
    if side_data is None:
        load_errors.append((path, f'missing side {side_key}'))
        continue

    entry = {**{k: None for k in joint_map}, **{k: None for k in force_map}}
    for metric_name, key_path in joint_map.items():
        raw = get_nested(side_data, key_path + ['mean'])
        entry[metric_name] = ensure_array(raw)
    for force_name, key_path in force_map.items():
        raw = get_nested(side_data, key_path + ['mean'])
        entry[force_name] = ensure_array(raw)

    if not any(v is not None for v in entry.values()):
        load_errors.append((path, 'no metrics found'))
        continue

    speed_groups.setdefault(speed, []).append(entry)

    # Save this individual trial as its own reference file.
    if all(entry[k] is not None for k in ['hip_flexion', 'knee_angle', 'Fx', 'Fy']):
        parts = path.split(os.sep)
        participant = parts[-2] if len(parts) >= 2 else 'unknown'
        group = parts[-3] if len(parts) >= 3 else 'unknown'
        out = {
            'hip': {'mean': np.deg2rad(entry['hip_flexion'])},
            'knee': {'mean': -np.deg2rad(entry['knee_angle'])},
            'Fx': {'mean': entry['Fx'] / 9.81},
            'Fy': {'mean': entry['Fy'] / 9.81},
        }
        if entry['ankle_angle'] is not None:
            out['ankle'] = {'mean': np.deg2rad(entry['ankle_angle'])}
        speed_str = f'{speed:.2f}'.replace('.', '_')
        out_name = f'{group}_{participant}_{speed_str}.mat'
        scipy.io.savemat(os.path.join(walking_processed_dir, out_name), out)
        n_saved += 1

print(f'Loaded speed groups: {sorted(speed_groups.keys())}')
print(f'Load errors: {len(load_errors)}')
print(f'Saved {n_saved} individual trial files to {walking_processed_dir}')


In [ ]:
summary_rows = []
grouped = {}
output_base = 'reference_data'
os.makedirs(output_base, exist_ok=True)

for speed, entries in sorted(speed_groups.items()):
    grouped[speed] = {}
    for metric in list(joint_map) + list(force_map):
        series_list = [e[metric] for e in entries if e[metric] is not None]
        if not series_list:
            continue
        lengths = [len(s) for s in series_list]
        target_len = int(np.median(lengths)) if lengths else 100
        obs = [resample_series(s, target_len) for s in series_list]
        stacked = np.vstack(obs)
        mean_arr = np.mean(stacked, axis=0)
        var_arr = np.var(stacked, axis=0)
        grouped[speed][metric] = {'mean': mean_arr, 'var': var_arr, 'n': len(obs)}
        df = pd.DataFrame({'mean': mean_arr, 'var': var_arr})
        speed_str = str(speed).replace('.', '_')
        out_csv = os.path.join(output_base, f'speed_{speed_str}_{metric}.csv')
        df.to_csv(out_csv, index=False)
        summary_rows.append({'speed': speed, 'metric': metric, 'n_participants': len(obs), 'csv_path': out_csv})

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(output_base, 'gait_speed_summary.csv'), index=False)
npz_path = os.path.join(output_base, 'gait_speed_mean_var.npz')
npz_data = {}
for speed, metrics in grouped.items():
    s = str(speed).replace('.', '_')
    for m, stats in metrics.items():
        npz_data[f'{s}_{m}_mean'] = stats['mean']
        npz_data[f'{s}_{m}_var'] = stats['var']
np.savez_compressed(npz_path, **npz_data)
print(f'Saved results to {output_base}')
